# CivicWater Sentinel — P2 · Step 2: XGBoost per sensor, served with Snap ML

Step 2 of the model ladder (design doc §3, Table 2): **one XGBoost model per pressure sensor**,
predicting its pressure from inlet flows, tank level, time and the *other* sensors at the same
instant. The pattern follows the organisers' *Loan Approval* demo (XGBoost GBTree for the
structured prediction) and their "train anywhere, infer on IBM Z" flow:

1. **Part A – train (off-platform, e.g. Colab)**: fit, validate, export portable XGBoost JSON.
2. **Part B – on LinuxONE**: import the JSON into **Snap ML**, which runs the trees on the Telum
   AI accelerator (zDNN) when available; check x86↔s390x parity and measure latency.

TreeSHAP contributions (`pred_contribs`) explain each forecast for the operator (design doc §6).

**Running in Colab (training off-platform, design doc §4).** Clone the repository, upload
`scada_2018.csv.gz`, `sensors.csv` and `leaks_2018.csv` from `shared/civicwater/data/v1/`, then
run this cell before the setup cell (the registry defaults to the repo's official
`config/sensors.battledim.json`, as on the VM):

```python
!git clone https://github.com/pedruck/Team-CodigoMaster_IBM-Z-Datathon-2026---Civic-Water-Sentinal.git /content/repo
!pip -q install xgboost onnx onnxscript torch
import os
os.environ["REPO"] = "/content/repo"
os.environ["SENTINEL_DATA_DIR"] = "/content/data"      # put the v1/ files in /content/data/v1/
os.environ["SENTINEL_MODELS_DIR"] = "/content/models"  # copy this folder to shared/civicwater/models/ on the VM
```

In [ ]:
# --- Setup (same for every P2 notebook) --------------------------------------------------
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")

# Repository root = first parent folder that contains src/sentinel (works on the VM and in Colab)
REPO = Path(os.environ.get("REPO") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sentinel").exists()))
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sentinel.forecasting import training as p2lib   # training helpers (tested in CI)
FAST = os.environ.get("P2_FAST") == "1"   # smoke-test mode: smaller training, same code path
MASK_POLICY = "reparados"   # P1's training mask (docs/data.md §7): repaired 2018 leaks excluded
print("repo   :", REPO)
print("data   :", p2lib.data_dir())
print("models :", p2lib.models_dir())
print("sensor registry:", os.environ.get("SENTINEL_SENSOR_REGISTRY") or p2lib.DEFAULT_REGISTRY, "| mask policy:", MASK_POLICY)
print("on LinuxONE (s390x):", p2lib.on_linuxone(), "| FAST mode:", FAST)

# Part A — Training

In [ ]:
import xgboost as xgb
layout = p2lib.load_layout()
df = p2lib.load_scada(2018, layout)
P = list(layout.pressure)
free = p2lib.clean_mask(df.index, MASK_POLICY)
train, val = p2lib.split(df)
print(df.shape, "| xgboost", xgb.__version__, f"| clean windows ({MASK_POLICY}): {free.mean():.1%}")

## Train/serve parity of the features

The features are defined once, in `sentinel/forecasting/features.py`, and used by the API. The
vectorised version used for training is compared with it on 200 random rows; any difference
stops the notebook.

In [ ]:
n = p2lib.assert_feature_parity(df, layout, n=50 if FAST else 200)
print(f"feature parity OK on {n} rows × {len(P)} sensors")

## Train 33 models (Jan–Oct), early-stopped on Nov–Dec

Rows outside P1's clean mask are dropped from training. Missing inputs stay `NaN`
(XGBoost handles them); rows with a missing target are dropped.

In [ ]:
PARAMS = dict(n_estimators=60 if FAST else 600, learning_rate=0.05, max_depth=6, subsample=0.8,
              colsample_bytree=0.8, tree_method="hist", early_stopping_rounds=30)
tr = train[free.loc[train.index]]
models, best_iter, val_pred = {}, {}, {}
for i, s in enumerate(P, 1):
    Xtr, ytr = p2lib.xgb_frame(tr, layout, s), tr[s]
    Xva, yva = p2lib.xgb_frame(val, layout, s), val[s]
    ok_tr, ok_va = ytr.notna(), yva.notna()
    m = xgb.XGBRegressor(**PARAMS)
    m.fit(Xtr[ok_tr], ytr[ok_tr], eval_set=[(Xva[ok_va], yva[ok_va])], verbose=False)
    models[s], best_iter[s] = m, int(m.best_iteration) + 1
    val_pred[s] = pd.Series(m.predict(Xva), index=val.index)
    if i % 11 == 0 or i == len(P):
        print(f"{i}/{len(P)} sensors trained")
resid = val[P] - pd.DataFrame(val_pred)
mae, sigma = p2lib.mae_by_sensor(resid), p2lib.robust_sigma(resid)
contrast = p2lib.leak_contrast(resid, sigma, free)
print(f"validation MAE: {mae.mean():.3f} m | leak contrast: {contrast}")
print("low-variance sensors (σ < 0.02 m):", p2lib.low_variance_sensors(sigma))

In [ ]:
# Compare with step 1 (baseline) on the same validation period
base_path = p2lib.models_dir() / "baseline-hourofweek@0.1.0.json"
if base_path.exists():
    base_mae = pd.Series(p2lib.read_json(base_path)["validation"]["mae_m"])
    cmp = pd.DataFrame({"baseline": base_mae, "xgboost": mae}).dropna()
    print(f"mean MAE  baseline {cmp.baseline.mean():.3f} m  →  xgboost {cmp.xgboost.mean():.3f} m")
    cmp.plot.bar(figsize=(14, 3), title="Validation MAE per sensor (m)"); plt.show()
else:
    print("run 01_baseline.ipynb first to compare with the baseline")

## Refit on all clean 2018 windows and export portable JSON

Models cross the platform boundary **only as portable files** (design doc §4): binary XGBoost
models depend on byte order. XGBoost 3.x writes `base_score` as a vector, which Snap ML 1.17 on
the LinuxONE image cannot parse, so the export normalises it (`portable_booster_json`, lossless).
A parity sample (raw rows + x86 predictions) travels with the model for the check in Part B.

In [ ]:
from sentinel.forecasting.xgb import MODEL_NAME, portable_booster_json
VERSION = "0.1.0"
out = p2lib.models_dir() / f"{MODEL_NAME}@{VERSION}"
out.mkdir(parents=True, exist_ok=True)
full = df[free]
final = {}
for s in P:
    X, y = p2lib.xgb_frame(full, layout, s), full[s]
    ok = y.notna()
    params = {k: v for k, v in PARAMS.items() if k != "early_stopping_rounds"} | {"n_estimators": best_iter[s]}
    m = xgb.XGBRegressor(**params).fit(X[ok], y[ok], verbose=False)
    booster = m.get_booster()
    booster.feature_names = list(X.columns)
    raw = bytes(booster.save_raw("json")).decode()
    (out / f"{s}.json").write_text(portable_booster_json(raw))
    final[s] = m

sample = p2lib.complete_rows(val, layout).sample(64, random_state=3).sort_index()
expected = pd.DataFrame({s: final[s].predict(p2lib.xgb_frame(sample, layout, s)) for s in P}, index=sample.index)
sample.to_csv(out / "parity_rows.csv")
expected.to_csv(out / "parity_expected_x86.csv")

meta = p2lib.artefact_meta(
    MODEL_NAME, VERSION, layout, sensors=P,
    feature_names={s: p2lib.xgb_frame(sample.iloc[:1], layout, s).columns.tolist() for s in P},
    params=PARAMS, best_iteration=best_iter,
    trained_on="2018 clean windows (P1 mask); trees early-stopped on Nov-Dec 2018",
    validation={"period": "2018-11-01..2018-12-31", "mae_m_mean": round(float(mae.mean()), 4),
                "mae_m": mae.round(4).to_dict(), "leak_contrast": contrast},
    sigma_m=sigma.round(5).to_dict(), mask_policy=MASK_POLICY,
)
p2lib.write_json(out / "meta.json", meta)
size = sum(f.stat().st_size for f in out.iterdir()) / 1024**2
print(f"exported {len(P)} boosters + meta + parity sample to {out} ({size:.1f} MB)")

## Explainability: TreeSHAP contributions as in the alert payload

`pred_contribs=True` gives, per feature, how much it moved the forecast. Grouped as in the alert
payload (base, inlet flow, tank level, hour of week, other sensors), they **add up to the
forecast** (design doc §6).

In [ ]:
from sentinel.forecasting.features import group_contributions
s = P[0]
X1 = p2lib.xgb_frame(sample.iloc[:1], layout, s)
contrib = final[s].get_booster().predict(xgb.DMatrix(X1), pred_contribs=True)[0]
groups = group_contributions(list(X1.columns), contrib[:-1], contrib[-1])
forecast = float(final[s].predict(X1)[0])
print(f"{s} at {X1.index[0]}: forecast {forecast:.3f} m = " +
      " + ".join(f"{k} {v:+.3f}" for k, v in groups.items()))
assert abs(sum(groups.values()) - forecast) < 1e-3

# Part B — On IBM LinuxONE

Copy `models/xgboost-33@0.1.0/` to `shared/civicwater/models/` on the VM and run from here.
The cells also run on x86 (Snap ML then uses compressed trees on the CPU), which is useful as a
dry run, but the accelerator numbers only mean something on s390x.

## IBM Z AI accelerator check

Same checks as the organisers' notebooks, plus Snap ML.

In [ ]:
report = p2lib.accelerator_report()
for k, v in report.items():
    print(f"{k:16}: {v}")
if not report["nnpa_in_cpuinfo"]:
    print("\nNNPA not exposed: Snap ML falls back to compressed trees on the CPU ('accelerator-ready').")

## Load the model exactly as the API does

`tree_format="auto"`: Snap ML chooses `zdnn_tensors` (Telum AI accelerator) when available,
`compress_trees` (CPU) otherwise. The chosen format is read back from the model.

In [ ]:
from sentinel.forecasting.xgb import XGBoostForecaster
model_dir = p2lib.models_dir() / "xgboost-33@0.1.0"
model = XGBoostForecaster(model_dir, engine="snapml", tree_format="auto")
print(model.info, "| Snap ML tree format in use:", model.tree_format_used)

## x86 ↔ s390x parity

The rows exported in Part A are turned into API windows and scored here. Byte-order problems
would show up as large differences. The accelerator computes in reduced precision, so a small
difference is expected with `zdnn_tensors`; it is reported, not hidden.

In [ ]:
rows = pd.read_csv(model_dir / "parity_rows.csv", index_col="timestamp", parse_dates=True)
ref = pd.read_csv(model_dir / "parity_expected_x86.csv", index_col="timestamp", parse_dates=True)
native = XGBoostForecaster(model_dir, engine="xgboost", with_contributions=False)
from sentinel.schemas.forecasts import ForecastRequest
diffs = {"snapml": [], "xgboost-native": []}
for ts, row in rows.iterrows():
    w = p2lib.row_to_window(ts, row, layout)
    req = ForecastRequest(target_timestamp=w.timestamp, pressure_sensor_ids=tuple(P), history=(w,), current=w)
    for name, m in (("snapml", model), ("xgboost-native", native)):
        out_ = m.predict(req).expected_pressures
        diffs[name].append(max(abs(out_[s] - ref.loc[ts, s]) for s in P))
for name, d in diffs.items():
    print(f"{name:15}: max |s390x − x86| = {max(d):.2e} m over {len(rows)} windows × {len(P)} sensors")

## Latency per 5-minute window (33 sensors)

Same benchmark style as the organisers' notebooks (median / best / p99). The API scores one
window at a time, so this is the realistic number for the alert path.

In [ ]:
ts0, row0 = next(rows.iterrows())
w0 = p2lib.row_to_window(ts0, row0, layout)
req0 = ForecastRequest(target_timestamp=w0.timestamp, pressure_sensor_ids=tuple(P), history=(w0,), current=w0)
engines = {
    "xgboost native (CPU)": XGBoostForecaster(model_dir, engine="xgboost", with_contributions=False),
    "snapml compress_trees (CPU)": XGBoostForecaster(model_dir, engine="snapml", tree_format="compress_trees", with_contributions=False),
}
try:
    engines["snapml zdnn_tensors (NNPA)"] = XGBoostForecaster(model_dir, engine="snapml", tree_format="zdnn_tensors", with_contributions=False)
except Exception as exc:
    print("zdnn_tensors not available here:", type(exc).__name__, str(exc)[:120])
engines["snapml auto + TreeSHAP (API default)"] = model
lat = {name: p2lib.benchmark(lambda m=m: m.predict(req0), repeats=30 if FAST else 100) for name, m in engines.items()}
lat_df = pd.DataFrame(lat).T.round(3)
lat_df

In [ ]:
# Batch throughput, as in the organisers' benchmark cells (one sensor, 1000 rows)
s = P[0]
Xb = p2lib.xgb_frame(p2lib.complete_rows(val, layout).iloc[:1000], layout, s).to_numpy(np.float32)
for name, m in engines.items():
    if m._engine == "snapml":
        r = p2lib.benchmark(lambda m=m: m._models[s].predict(Xb), repeats=20)
    else:
        r = p2lib.benchmark(lambda m=m: m._booster_predict(s, Xb, False), repeats=20)
    print(f"{name:38}: {r['p50_ms']:8.2f} ms / 1000 rows  →  {1000 / (r['p50_ms'] / 1000):>12,.0f} rows/s")

In [ ]:
meta = p2lib.read_json(model_dir / "meta.json")
meta.setdefault("latency_ms", {})[report["machine"]] = {k: v for k, v in lat_df.T.to_dict().items()}
meta.setdefault("parity", {})[report["machine"]] = {k: float(max(v)) for k, v in diffs.items()}
p2lib.write_json(model_dir / "meta.json", meta)
print("latency and parity recorded in meta.json for", report["machine"])

## Summary

* Validation MAE vs the baseline is in Part A; latency and parity per machine are stored in
  `meta.json`, so the ladder comparison in notebook 03 reads them back.
* The API loads this model with `load_forecaster("xgboost", <dir>)`; forecasts carry TreeSHAP
  contributions grouped as in the alert payload.
* Next: `03_gru_keras_zdnn.ipynb`.